# PD Controllers (PD_1 and PD_2)

Cheat sheet for some of the parameters symbols
α  β  ρ  θ  ξ  δ
₀ ₁ ₂ ₃ ₄ ₅ ₆ ₇ ₈ ₉

**Transfer function (PD_1, crossed):**
G_c(s) = [α₁·s + (α₁−β·α₂)] / (s+1)²

**Key gains:**
- K_P^PD = α₁ − β·α₂  (proportional-like coefficient)
- K_D^PD = α₁          (derivative coefficient)
- G_eff = (α₁ − β·α₂)·θ₁·θ₂

**Steady-state error:** e_ss = r / (1 + G_eff)  — always non-zero (no integrator).

**PD_1 (crossed):** β increases → G_eff decreases → more damping, larger SS error, expanded stable region.
**PD_2 (uncrossed):** G_eff = (α₁ + β·α₂)·θ₁·θ₂ → β increases effective gain (destabilising).

**Stability boundary (Routh–Hurwitz, 3rd-order):** G_eff < **8**.

## Setup

In [63]:
# while building the package so it refreshes modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [64]:
import numpy as np
import sympy as sp
import warnings
warnings.filterwarnings('ignore')
from bokeh.io import output_notebook, show

import biomolecular_controllers as bc
from biomolecular_controllers.gain import pd1_stability_boundary, pd2_stability_boundary

print('✓ Imports successful!')
from biomolecular_controllers.figure_saver import save_fig
from biomolecular_controllers.figure_gifs import save_gif


output_notebook()


✓ Imports successful!


Loading BokehJS ...

In [65]:
output_notebook()

runner  = bc.SimulationRunner()
calc    = bc.MetricsCalculator()
stab    = bc.StabilityAnalyzer()
plotter = bc.VisualizationPipeline()
swiffer = bc.SensitivityAnalyzer()


print('✓ Tools initialized!')

Loading BokehJS ...

✓ Tools initialized!


## Parameter Sweep

In [66]:
# Set fixed params such that contribution to any gain(s) not being investigated is ~0.1 or less
# Effective Gain K_p = alpha1 + alpha2*beta  
# Effective Gain K_d = alpha1
# Stability: 8*(-beta*alpha2/alpha1 + sqrt((beta*alpha2/alpha1)^2 + 1)) > alpha1*theta1*theta2

alpha1, alpha2, beta, theta1, theta2 = sp.symbols(
    'alpha1 alpha2 beta theta1 theta2', positive=True
)

K1 = alpha1 * theta1 * theta2
K2 = alpha2 * beta * theta1 * theta2

# Routh-Hurwitz conditions, each must be > 0
rh_conditions = {
    'P_LPF': 8 - K1,                    # upper bound, same as P controller
    'PD_LPF':    8 * (-beta*alpha2/alpha1 + 
             sp.sqrt((beta*alpha2/alpha1)**2 + 1)) - K1 
}

base_params = {
    theta1: 1.0,    
    theta2: 1.0,    # theta1*theta2 = 1 so gains read directly
    alpha1: 1.0,    # fix proportional, isolate derivative effect of beta
    alpha2: 0.1,    # small so derivative correction is minor at beta=0
    beta:   1.0,    # initialization, overridden when sweeping
}


# example with simulation parameters
# sweep alpha1, fix everything else
lower, upper = swiffer.get_feasible_range(rh_conditions, beta, base_params)

inside, out_low, out_high, combined = swiffer.build_sweep(lower, upper)
print(f"Feasible range for beta with small excursion above/below stable regime: ({lower:.4f}, {upper:.4f})")
print(f"Inside:  {np.round(inside, 3)}")
print(f"Below:   {np.round(out_low, 3)}")
print(f"Above:   {np.round(out_high, 3)}")

  PD_LPF: upper bound at 39.3750
Feasible range for beta with small excursion above/below stable regime: (0.0000, 39.3750)
Inside:  [ 1.969  7.031 12.094 17.156 22.219 27.281 32.344 37.406]
Below:   []
Above:   [45.281]


## Part 1: Sweep β (Derivative Strength) — PD_1

Hold α₁ near the PC stability boundary (α₁=7.9 ≈ G_crit=8 with θ₁=θ₂=1).
As β increases, G_eff = (α₁ − β·α₂)·θ₁·θ₂ decreases → circuit moves away from instability.

**e_ss = r / (1 + G_eff)**  where G_eff = (α₁ − β·α₂)·θ₁·θ₂

In [67]:
# Fixed parameters
alpha_1 = 7.9    # near G_crit boundary
alpha_2 = 1.0
theta_1 = 1.0
theta_2 = 1.0

# Sweep beta: 0 = pure PC, alpha_1/alpha_2 = zero gain
beta_vals = np.linspace(0.0, 5.0, 8)

# Step input
ref_low  = bc.DEFAULT_PARAMS['PD_1']['ref']
ref_high = 50.0
t_step   = 150.0
t_span   = (0, 500)

print(f'Sweeping β: {beta_vals.min():.1f} → {beta_vals.max():.1f}')
print(f'Step input: {ref_low} → {ref_high} at t={t_step}')
print(f'G_eff range: {(alpha_1 - beta_vals[-1]*alpha_2)*theta_1*theta_2:.2f} – {(alpha_1 - beta_vals[0]*alpha_2)*theta_1*theta_2:.2f}')

Sweeping β: 0.0 → 5.0
Step input: 10.0 → 50.0 at t=150.0
G_eff range: 2.90 – 7.90


In [68]:
trajectories_dict = {}
gains_list        = []
ss_errors_list    = []
ss_stds_list      = []
settling_times_list = []
overshoots_list   = []
rise_times_list   = []

params = {
        'alpha_1': alpha_1,
        'alpha_2': alpha_2,
        'theta_1': theta_1,
        'theta_2': theta_2,
        'beta':    beta,
    }

print('Running PD_1 simulations...')

for beta in beta_vals:
    current_params = {**params, 'beta': beta}
    perturbations = [
        {'time': t_step, 'type': 'parameter', 'param': 'ref', 'value': ref_high}
    ]
    try:
        result = runner.run_with_perturbations(
            model='PD_1', t_span=t_span, points=1000,
            perturbations=perturbations, params=current_params
        )
        time = np.asarray(result['time'],           dtype=float)
        y    = np.asarray(result['states']['y'],    dtype=float)
        ref  = np.where(time < t_step, ref_low, ref_high)

         # Effective Closed-loop gain: G = (α₁ − β·α₂)·θ₁·θ₂
        K_D_eff_num = bc.gain.gain_pd1(current_params)
        
        # PD has SS error — use actual achieved SS as the rise-time target.
        # Using ref_high directly causes the 90% threshold to exceed the actual
        # steady-state value, making rise_time invalid for most gain settings.
        y_ss = float(np.mean(y[-20:]))

        os_r = calc.overshoot(time, y, ref=ref_high, pert_start=t_step, pert_end=t_span[1])
        st_r = calc.settling_time(time, y, ref=ref_high, pert_start=t_step, pert_end=t_span[1])
        rt_r = calc.rise_time(time, y, ref=y_ss, pert_start=t_step, pert_end=t_span[1])
        ss_r = calc.steady_state(time,y, pert_start=t_step, pert_end=t_span[1])

        trajectories_dict[beta] = {'time': time, 'y': y, 'ref': ref}
        gains_list.append(K_D_eff_num)
        ss_errors_list.append(abs(ss_r['ss_value'] - ref_high))
        ss_stds_list.append(ss_r['ss_std'])
        settling_times_list.append(st_r['settling_time'] if st_r['settled'] else np.nan)
        overshoots_list.append(os_r['magnitude'])
        rise_times_list.append(rt_r['rise_time'] if rt_r['valid'] else np.nan)
    except Exception as e:
        print(f'  β={beta:.2f}: failed ({str(e)[:40]})')
        trajectories_dict[beta] = None
        gains_list.append((alpha_1 - beta * alpha_2) * theta_1 * theta_2)
        ss_errors_list.append(np.nan)
        settling_times_list.append(np.nan)
        overshoots_list.append(np.nan)
        rise_times_list.append(np.nan)

gains           = np.array(gains_list)
ss_errors       = np.array(ss_errors_list)
ss_stds         = np.array(ss_stds_list)
settling_times  = np.array(settling_times_list)
overshoots      = np.array(overshoots_list)
rise_times      = np.array(rise_times_list)

# Remove None trajectories for plotter
trajectories_dict = {k: v for k, v in trajectories_dict.items() if v is not None}
print(f'✓ Completed {len(trajectories_dict)}/{len(beta_vals)} simulations')

Running PD_1 simulations...
✓ Completed 8/8 simulations


## Plot 1: Steady-State Error vs G_eff

PD has non-zero SS error: **e_ss = r / (1 + G_eff)** where G_eff = (α₁−β·α₂)·θ₁·θ₂.
Higher β → lower G_eff → larger SS error (tradeoff: stability for accuracy).

In [69]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=beta_vals,
    trajectories=trajectories_dict,
    metric_name='Steady State Error',
    gain_vals=gains,
    metric_vals=ss_errors,
    metric_stds=ss_stds,
    title='PD_1: Steady-State Error vs K_D  [e_ss = r/(1+(α₁−β·α₂)·θ₁·θ₂)]',
    x_label='K_D = (α₁−β·α₂)·θ₁·θ₂',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PD_1', 'steady_state_error', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PD_1', 'steady_state_error')



[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pd_1_steady_state_error.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pd_1_steady_state_error.gif')

## Plot 2: Settling Time vs G_eff

In [70]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=beta_vals,
    trajectories=trajectories_dict,
    metric_name='Settling Time',
    gain_vals=gains,
    metric_vals=settling_times,
    title='PD_1: Settling Time vs K_D  [e_ss = r/(1+(α₁−β·α₂)·θ₁·θ₂)]',
    x_label='K_D = (α₁−β·α₂)·θ₁·θ₂',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PD_1', 'settling_time', fmt='png')
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PD_1', 'settling_time')

[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pd_1_settling_time.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pd_1_settling_time.gif')

## Plot 3: Overshoot vs G_eff

In [71]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=beta_vals,
    trajectories=trajectories_dict,
    metric_name='Overshoot',
    gain_vals=gains,
    metric_vals=overshoots,
    title='PD_1: Overshoot vs K_D  [e_ss = r/(1+(α₁−β·α₂)·θ₁·θ₂)]',
    x_label='K_D = (α₁−β·α₂)·θ₁·θ₂',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PD_1', 'overshoot', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PD_1', 'overshoot')


[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pd_1_overshoot.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pd_1_overshoot.gif')

## Plot 4: Rise Time vs G_eff

In [72]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=beta_vals,
    trajectories=trajectories_dict,
    metric_name='Rise Time',
    gain_vals=gains,
    metric_vals=rise_times,
    title='PD_1: Rise Time vs K_D  [e_ss = r/(1+(α₁−β·α₂)·θ₁·θ₂)]',
    x_label='K_D = (α₁−β·α₂)·θ₁·θ₂',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PD_1', 'rise_time', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PD_1', 'rise_time')


[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pd_1_rise_time.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pd_1_rise_time.gif')

## Part 2: Stability Diagrams (α₁ vs θ₁)

Boundary: θ₁ = G_crit / (G_eff · θ₂)
- **PD_1**: G_eff = α₁ − β·α₂  →  boundary shifts *right* with higher β (larger stable region)
- **PD_2**: G_eff = α₁ + β·α₂  →  boundary shifts *left* with higher β (smaller stable region)

In [73]:
# PD_1 Parameter stability diagram
alpha_range = np.linspace(0.1, 10, 400)
theta_range = np.linspace(0.1, 10, 400)
theta_2_fixed = 1.0

boundary_fn = bc.gain.pd1_stability_boundary(theta_2=theta_2_fixed, alpha_2=0.1, beta=1.0)

p = plotter.plot_stability_diagram(
    x_vals=alpha_range,
    y_vals=theta_range,
    stable_condition=lambda A, T: (A * T * theta_2_fixed < 8),
    boundary_fns=[
        {
            'fn':    boundary_fn,
            'label': 'PD_1 boundary  (G_eff=8)',
            'color': 'black',
            'dash':  'dashed',
            'line_width': 2.0,
        },
    ],
    x_name='α₁',
    y_name='θ₁',
    title=f'PD_1 Parameter Stability Diagram  (θ₂={theta_2_fixed})',
)
p.scatter([-999], [-999], marker="square", size=14,
          fill_color="#8ECF8E", fill_alpha=0.65,
          line_color=None, legend_label="Stable")
p.scatter([-999], [-999], marker="square", size=14,
          fill_color="#C8C8C8", fill_alpha=0.65,
          line_color=None, legend_label="Unstable")

save_fig(p, 'PD_1', 'stability_diagram', fmt='png', show=True)

  Saved PNG  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pd_1_stability_diagram.png


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pd_1_stability_diagram.png')

## Part 3: Root Locus (Dominant Eigenvalue, PD_1)

Sweep α₁ from 0 upward.  Increasing α₁ pushes eigenvalues toward the imaginary axis; derivative action (β) provides extra damping compared to pure PC.

In [74]:
root_alpha_vals = np.linspace(0.1, 5.5, 12)

eigenvalue_paths = {}
for model_key in ['PD_1', 'PD_2']:
    reals, imags, gs = [], [], []
    for val in root_alpha_vals:
        p_params = {
            'alpha_1': val,
            'alpha_2': 2.0,
            'theta_1': 1.0,
            'theta_2': 1.0,
            'beta':    1.0,
        }
        try:
            res  = stab.analyze_stability(model=model_key, params=p_params, steady_state_time=100.0)
            eigs = res.eigenvalues
            dom  = eigs[np.argmax(np.real(eigs))]
            reals.append(float(np.real(dom)))
            imags.append(float(np.imag(dom)))
            gs.append(res.gain)
        except Exception:
            reals.append(np.nan); imags.append(np.nan); gs.append(np.nan)
    eigenvalue_paths[model_key] = {
        'real': np.array(reals),
        'imag': np.array(imags),
        'gain': np.array(gs),
    }

fig = plotter.plot_root_locus(
    eigenvalue_paths,
    title='PD Controllers Root Locus: Dominant Eigenvalue as α₁ varies'
)

save_fig(fig, 'PD_controllers', 'root_locus', fmt='png', show=True)


  Saved PNG  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pd_controllers_root_locus.png


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pd_controllers_root_locus.png')